# Quantum walks

This notebook summarizes the Szegedy walk and its logical circuit resources in `monaqa2.qiskit2`. The depth bounds below are those tested in [3.3 resource validation](<3.3_validation_resources(1).ipynb>); the component tests and estimation helpers remain there.

1. [Szegedy quantum walk operator](#szegedy-quantum-walk-operator)
2. [Summary of the resources](#summary-of-the-resources)
3. [Primitives](#primitives)
4. [Boltzmann coin](#boltzmann-coin)
5. [Proposal preparation unitaries](#proposal-preparation-unitaries)
6. [Reflection and accept-path unitary](#reflection-and-accept-path-unitary)
7. [Complete walk](#complete-walk)


<a id="szegedy-quantum-walk-operator"></a>
## Szegedy quantum walk operator

For a symmetric proposal matrix $T$ on $n$ spins, the Metropolis chain has off-diagonal entries

$$
(P_\beta)_{yx}=T_{yx}A_{yx}^{(\beta)},\qquad
A_{yx}^{(\beta)}=\min\{1,e^{-\beta[H(y)-H(x)]}\},\qquad y\ne x,
$$

with diagonal entries fixed by column normalization. Here $\beta\ge0$ is the inverse temperature. The implemented coin uses the coefficient-rounded Hamiltonian at the precision specified below.

The current and proposed states occupy registers $A$ and $B$. Write $c$ for the persistent coin register and $p$ for any extra proposal wires. Clean arithmetic work is implicit. With proposal preparation $V$, Boltzmann coin $B_\beta$, and accepted swap $F_{\rm acc}$, one walk application is

$$
W_\beta=R_0V^\dagger B_\beta^\dagger F_{\rm acc}B_\beta V.
$$

Acceptance is the all-zero state of $c$: $F_{\rm acc}$ swaps $A$ and $B$ exactly on that subspace. The reflection is

$$
\Pi_0=I_A\otimes|0\rangle\!\langle0|_{B,c,p},\qquad R_0=2\Pi_0-I.
$$

For the four proposals below, projecting the input and output of $W_\beta$ onto zero in $B,c,p$ gives the discriminant matrix

$$
(X_\beta)_{xy}=\sqrt{(P_\beta)_{xy}(P_\beta)_{yx}}.
$$

For a reversible chain, $X_\beta$ and $P_\beta$ have the same spectrum; a discriminant eigenvalue $\lambda$ gives walk eigenvalues $e^{\pm i\arccos\lambda}$. For the implemented circuit, the chain here uses the rounded coefficients. Symmetry of each Trotterized Hamiltonian evolution ensures its proposal phases cancel in this block.


<a id="summary-of-the-resources"></a>
## Summary of the resources

Depth means the `get_nc_depth` metric from 3.3: layers of $T$, $T^\dagger$, and non-Clifford $R_z$ after explicit expansion into Clifford+$R_z$. Connectivity is all-to-all; arbitrary logical rotations count as one layer. Clifford fanout has zero depth in this metric, although its circuit depth is logarithmic.

The variables used below are:

- $n\ge1$: number of spins; $M=n(n+1)/2$: number of field and pair-term slots, including zero coefficients.
- $W=\texttt{walk.coin.W}\ge2$: actual signed energy-word width; $C=W-1$: number of persistent coin qubits.
- $r\ge1$: Trotter steps; $k\ge1$: bits per randomized parameter, giving $2^k$ values for each of $t$ and $\gamma$.
- $E$: extra proposal wires reflected with $B,c$; $E=2kn$ for randomized Hamiltonian proposals and zero otherwise.

The qubit column counts each component as a standalone gate, including its two system registers and clean work. These counts are not added when composing the walk: clean work is reused.

| Component | simple upper bound depth | asymp | Logical qubits |
|---|---:|---|---:|
| ProposalUniform | $0$ | $0$ | $2n$ |
| ProposalLocal | $2\lceil\log_2 n\rceil$ | $O(\log n)$ | $2n$ |
| ProposalHamSim | $r(n+2)+1$ | $O(rn)$ | $2n$ |
| ProposalHamSimRandomized | $(1+4k+8k^2)[r(n+2)+1]$ | $O(rnk^2)$ | $2n+2kn$ |
| BoltzmannCoin | $36\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+24\lceil\log_2 W\rceil+44$ | $O(\log n+\log W)$ | $O(n^2W+W\log W)$ |
| AcceptPath | $12\lceil\log_2 C\rceil+3$ | $O(\log C)$ | $O(n+C)$ |
| Reflection | $12\lceil\log_2(n+C+E)\rceil-6$ | $O(\log(n+C+E))$ | $O(n+C+E)$ |


<a id="primitives"></a>
## Primitives

$m$ is the number of input qubits in a zero test, excluding its flag and work. For fanout it is the total number of wires: one source and $m-1$ copies.

| Element | simple upper bound depth | asymp |
|---|---:|---|
| Ccx | $3$ | $O(1)$ |
| Rx, Ry, Rzz | $1$ | $O(1)$ |
| Crx, Cry, Crz, Crzz | $2$ | $O(1)$ |
| Ccrx, Ccry, Ccrz, Ccrzz | $8$ | $O(1)$ |
| GivensRotation | $2$ | $O(1)$ |
| ControlledGivensRotation | $14$ | $O(1)$ |
| ReversibleFanout | $0$ | $0$ |
| ReversibleZeroFlag | $6\lceil\log_2 m\rceil$ | $O(\log m)$ |


<a id="boltzmann-coin"></a>
## Boltzmann coin

`BoltzmannCoin(n, h, J, F, beta)` compares both full configurations through `ReversibleDeltaEnergy`; it supports all four proposals, including moves that change several spins. Conditional loading, Wallace compression, and a carry-lookahead adder compute the signed energy difference. `ReversiblePositivePart` keeps its nonnegative part.

Let $u=\texttt{coin.energy_unit}$ and let $S$ be the signed integer such that the rounded physical difference is $uS$. The nonnegative integer $z=\max(0,S)=\sum_{j=0}^{C-1}z_j2^j$ controls one rotation per coin qubit:

$$
\theta_j=2\arccos(e^{-\beta u2^j/2}),\qquad
\prod_{j:z_j=1}\cos(\theta_j/2)=e^{-\beta uz/2}.
$$

The controlled rotations act on distinct qubits in parallel, giving rotational-acceptance depth at most $2$. Projecting all $C=W-1$ coins onto zero therefore has probability $\min(1,e^{-\beta uS})$. The positive-part and delta-energy circuits are then inverted, returning all arithmetic work to zero. The coins remain available to the accepted swap and are inverted later by $B_\beta^\dagger$.

`Walk` chooses the coefficient precision $F$ from `eps`$=\varepsilon_b>0$:

$$
F=\max\!\left(0,\left\lceil\log_2\frac{\beta\|H\|_1n(n+1)}{4\varepsilon_b}\right\rceil\right),
\qquad
\|H\|_1=\sum_i|h_i|+\sum_{i<j}|J_{ij}|.
$$

Take $F=0$ when $\beta\|H\|_1=0$. Here $F$ is the fractional precision of the normalized coefficients; $W$ is determined from the resulting integer coefficients. All resource bounds use the actual $W$. This precision rule controls coefficient error; `eps` is not a bound on every source of walk error, such as Trotter error.


<a id="proposal-preparation-unitaries"></a>
## Proposal preparation unitaries

### Uniform proposal

`ProposalUniform(n)` applies Hadamards to $B$, preparing every proposed configuration with probability $2^{-n}$. It uses $2n$ qubits and zero non-Clifford depth.

### Local proposal

`ProposalLocal(n)` prepares a weight-one Dicke state with a balanced tree of Givens rotations, then applies parallel CNOTs from $A$ to $B$:

$$
|x\rangle_A|0^n\rangle_B\longmapsto
\frac{1}{\sqrt n}\sum_{|z|=1}|x\rangle_A|x\oplus z\rangle_B.
$$

It flips exactly one uniformly selected spin, with depth at most $2\lceil\log_2n\rceil$ and no qubits beyond $A,B$.

### Hamiltonian-simulation proposal

`ProposalHamSim` copies $x$ into $B$ and applies a symmetric second-order Trotter approximation to $e^{-itH_{\rm tf}(\gamma)}$, where

$$
H=\sum_i h_iZ_i+\sum_{i<j}J_{ij}Z_iZ_j,\qquad
H_{\rm tf}(\gamma)=(1-\gamma)H+\gamma H_X,\qquad H_X=-\sum_iX_i.
$$

With $\delta t=t/r$, the splitting is

$$
U^{(r)}(t,\gamma)=\left(
e^{-i\delta t\gamma H_X/2}
e^{-i\delta t(1-\gamma)H}
e^{-i\delta t\gamma H_X/2}
\right)^r.
$$

Adjacent $X$ half-steps merge. Each Ising layer has one parallel $Z$ layer and at most $n$ disjoint-pair $ZZ$ layers. Together with $r+1$ $X$ layers this gives $r(n+2)+1$ non-Clifford depth, using $2n$ qubits.

### Randomized Hamiltonian-simulation proposal

`ProposalHamSimRandomized` prepares a coherent uniform superposition over two $k$-bit midpoint grids for $t$ and $\gamma$. It applies the same evolution conditioned on those labels; the labels remain part of the walk space. Separate fanout copies keep rotations on different spins and disjoint pairs parallel. There are $2k$ label qubits and $2k(n-1)$ copies, giving $2n+2kn$ qubits in total. The controlled-rotation bound is $(1+4k+8k^2)[r(n+2)+1]$, as in 3.3.


<a id="reflection-and-accept-path-unitary"></a>
## Reflection and accept-path unitary

`AcceptPath(n, coins=C)` computes an all-zero coin flag, fans it out with a balanced CNOT tree, swaps all $n$ pairs in parallel, and uncomputes the flag and its copies. Its non-Clifford depth is at most $12\lceil\log_2C\rceil+3$; including Clifford gates adds the $O(\log n)$ fanout depth.

`Reflection(n, coins=C, extra=E)` implements $R_0=2\Pi_0-I$ by computing a balanced zero test on $B$, the coins, and all extra proposal wires, applying the phase, and uncomputing. Its non-Clifford depth is at most $12\lceil\log_2(n+C+E)\rceil-6$. Including proposal labels in this reflection is necessary for the randomized proposal.

Both gates return their clean work to zero and reuse the arithmetic workspace in `Walk`.


<a id="complete-walk"></a>
## Complete walk

`Walk(h, J, eps, beta, proposal="local", k=2, ...)` composes, in circuit order, $V$, $B_\beta$, $F_{\rm acc}$, $B_\beta^\dagger$, $V^\dagger$, $R_0$. The proposal choices are `"local"`, `"uniform"`, `"hamsim"`, and `"hamsim_randomized"`; Hamiltonian options include `t`, `gamma`, and `trotter_steps`$=r$.

Adding the component bounds gives the following simple upper bounds from 3.3:

| Proposal | simple upper bound depth | asymp |
|---|---:|---|
| Local / uniform | $72\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+76\lceil\log_2(n+W)\rceil+85$ | $O(\log n+\log W)$ |
| Hamiltonian | $2r(n+2)+72\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+72\lceil\log_2(n+W)\rceil+87$ | $O(rn+\log W)$ |
| Randomized Hamiltonian | $2(1+4k+8k^2)[r(n+2)+1]+72\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+72\lceil\log_2(n(2k+1)+W)\rceil+85$ | $O(rnk^2+\log W)$ |

The total qubit count is $O(n^2W+W\log W+E)$, with shared clean work allocated as the maximum needed by the coin, accepted swap, and reflection. The exact count is `walk.num_qubits`. Local and uniform walks retain logarithmic non-Clifford depth; Hamiltonian proposals add their Trotter depth.
